> **데이터셋 안내** — 이 노트북이 참조하는 HF 데이터셋은 공개 배포하지 않는다.
> AI Hub 원본에서 재생성하는 절차는 [docs/data/data-pipeline.md](../docs/data/data-pipeline.md)「가공 데이터셋은 배포하지 않는다 — 재현 경로」에 있다.

## 카디널리티 디코딩 — 문서별 기대-F1 plug-in

다라벨 문서 카디널리티를 추론 시 결정 규칙으로 회수 시도. 운영 채택 focal + **proper 손실 BCE 대조군**을 나란히 돌리고, 확률원을 **미보정(raw)·temperature·Platt** 세 가지로 갈라 카디널리티 신호가 어디에 있는지 국소화한다.

논거·프로토콜: `docs/experiments/cardinality-decoding.md`.

- **판정 축**: 멀티라벨 **micro-F1**, 오라클-k micro 상한 대비 실현율.
- **프로토콜**: 캘리브레이션·마진은 **val에서 적합, test에 1회 적용**.
- **게이트**: k≥2에서 Σp가 정답 평균(2.355)에 근접해야 2번째 양성 회수 가능.

In [1]:
import os
from dotenv import load_dotenv
from pathlib import Path
import json
import numpy as np

load_dotenv()

# 로컬
ROOT = Path(os.environ["DATA_ROOT"])
HF_HOME = ROOT / ".hf_cache"
os.environ["HF_HOME"] = str(HF_HOME)

from datasets import load_dataset
from sklearn.metrics import f1_score
from scipy.optimize import minimize_scalar, minimize

# 오류 분석 하니스에서 로짓·라벨 로딩 재사용(src/error_analysis.py — editable 설치 최상위 모듈)
from error_analysis import sigmoid, build_gold

In [2]:
# Config
config = {
    "seed": 42,
    "num_labels": 188,
    "tau": 0.5,
    "raw_ds": "ingyoun/patent-clean-text",
    "kmax": 10,                           # 문서당 카디널리티 상한(2~10개 Mno)
    "out_path": ROOT / "output",
}
# 운영 채택 focal + proper 손실 BCE 대조군(BCE는 γ 왜곡이 없어 확률이 카디널리티를 담는지 판별)
MODELS = [
    {"tag": "modernbert-patent-len512",     "kind": "focal"},
    {"tag": "modernbert-patent-len512-bce", "kind": "bce"},
]
CAL_NAMES = ["raw", "temp", "platt"]      # 확률원 — 미보정 · temperature · Platt
NUM, OUT = config["num_labels"], config["out_path"]
np.random.seed(config["seed"])

## 로짓·라벨 로딩 (val · test)

정답·카디널리티 축은 **모델 독립**(같은 문서) — 1회 로드. **로짓만 모델별.** val에서 적합, test에 1회 적용.

In [3]:
def load_labels(split):
    ds = load_dataset(config["raw_ds"], split=split)
    doc_ids = json.loads((OUT / f"doc_ids_{split}.json").read_text(encoding="utf-8"))
    assert ds["document_id"] == doc_ids, f"{split}: 로짓 행 순서 != 데이터셋 행 순서"
    return build_gold(ds["label_ids"], len(ds), NUM)

def load_logits(tag, split, n):
    L = np.load(OUT / f"logits_{tag}_{split}.npy")
    assert L.shape == (n, NUM), (tag, split, L.shape)
    return L

Yval, Ytest = load_labels("val"), load_labels("test")
kval, ktest = Yval.sum(1), Ytest.sum(1)
print(f"val  N={len(Yval):,}  k>=2 {(kval>=2).mean():.1%}")
print(f"test N={len(Ytest):,}  k>=2 {(ktest>=2).mean():.1%}")
print("모델:", [m["tag"] for m in MODELS])

val  N=11,162  k>=2 15.3%
test N=11,271  k>=2 15.0%
모델: ['modernbert-patent-len512', 'modernbert-patent-len512-bce']


## 1. 전역 캘리브레이션 (raw · temperature · Platt)

세 확률원을 모두 유지해 대조한다.

- **raw** `σ(z)` — 미보정.
- **temperature** `σ(z/T)` — 스케일만(부호 불변). `T<1`이면 sharpen(양성↑·음성↓), `T>1`이면 평탄화.
- **Platt** `σ(az+b)` — 스케일+이동. 전역 BCE(문서당 음성 ≈187개 지배)를 맞추느라 하향 이동(b<0)해 k≥2 양성 질량을 함께 누른다.

`T`·`a,b`는 val BCE 최소화로 적합.

In [4]:
def bce(P, Y):
    eps = 1e-7; P = np.clip(P, eps, 1 - eps)
    return float(-(Y * np.log(P) + (1 - Y) * np.log(1 - P)).mean())

def fit_calibration(Lval, Yval):
    T = float(minimize_scalar(lambda t: bce(sigmoid(Lval / t), Yval),
                              bounds=(0.2, 10.0), method="bounded").x)
    a, b = map(float, minimize(lambda p: bce(sigmoid(p[0] * Lval + p[1]), Yval),
                               x0=[1.0, 0.0], method="Nelder-Mead").x)
    CAL = {"raw":   lambda z: sigmoid(z),
           "temp":  lambda z: sigmoid(z / T),
           "platt": lambda z: sigmoid(a * z + b)}
    val_bce = {k: round(bce(CAL[k](Lval), Yval), 5) for k in CAL}
    return CAL, val_bce, {"temp_T": round(T, 4), "platt_a": round(a, 4), "platt_b": round(b, 4)}

## step 1 진단 — Σp vs 정답 k (세 확률원)

문서별 `Σp_c`가 정답 개수 `k`를 얼마나 복원하는가를 raw·temp·platt로 대조. raw Σp가 k≥2 정답(2.355)에 근접하고 platt이 멀면, 전역 BCE 캘리브레이션이 카디널리티 신호를 지운 것이다.

In [5]:
def sump_diag(logits, k, CAL):
    P = {cn: CAL[cn](logits) for cn in CAL}
    rec = {}
    for lab, sel in [("k=1", k == 1), ("k>=2", k >= 2)]:
        rec[lab] = {"n": int(sel.sum()), "mean_k_gold": round(float(k[sel].mean()), 3),
                    **{cn: round(float(P[cn].sum(1)[sel].mean()), 3) for cn in CAL}}
    return rec

## 2. 기대-F1 plug-in 디코더

- 라벨별 확률 `p_1≥p_2≥…` 정렬. 상위 `k` 집합의 기대 sample-F1 `E[F1(S_k)] ≈ 2·Σ_{i≤k}p_i / (k + Σ_c p_c)`를 `k=1..kmax`에 계산해 argmax `k`. 
- 기본 k=1, `E[F1(k)] - E[F1(1))` 가 val 마진을 넘을 때만 확장. baseline = τ=0.5(raw `logit≥0`)

In [6]:
def decode(logits, margin, prob_fn):
    P = prob_fn(logits)
    order = np.argsort(-P, axis=1)
    N, C = P.shape
    sump = P.sum(1)
    K = min(config["kmax"], C)
    idx = np.arange(1, K + 1)
    pred = np.zeros((N, C), dtype=bool)
    kchoice = np.ones(N, dtype=int)
    for i in range(N):
        ef1 = 2 * np.cumsum(P[i, order[i, :K]]) / (idx + sump[i])
        kb = int(ef1.argmax()) + 1
        if kb > 1 and ef1[kb - 1] - ef1[0] < margin:   # 가드: 마진 못 넘으면 k=1 유지
            kb = 1
        kchoice[i] = kb
        pred[i, order[i, :kb]] = True
    return pred, kchoice

def micro(Y, pred):
    return float(f1_score(Y, pred, average="micro", zero_division=0))

def oracle_k(logits, Y):
    """k≥2 문서만 정답 개수 k로 상위 k개 선택(랭킹 불변) — 도달 불가 micro 상한."""
    k = Y.sum(1); cf = (logits >= 0)
    for i in np.where(k >= 2)[0]:
        row = np.zeros(logits.shape[1], dtype=bool)
        row[np.argpartition(-logits[i], k[i])[:k[i]]] = True
        cf[i] = row
    return micro(Y, cf)

def slice_report(Y, pred, k):
    out = {}
    for lab, sel in [("k=1", k == 1), ("k>=2", k >= 2)]:
        fp, fn = int((pred & ~Y)[sel].sum()), int((Y & ~pred)[sel].sum())
        out[lab] = {"n": int(sel.sum()), "micro": round(micro(Y[sel], pred[sel]), 4),
                    "fp": fp, "fn": fn, "fp_fn": round(fp / max(fn, 1), 3),
                    "mean_pred": round(float(pred[sel].sum(1).mean()), 3)}
    return out

## 3. 모델별 파이프라인 (val 적합 → test 1회, 세 확률원)

한 모델에 대해 캘리브레이션 → 진단 → **세 확률원(raw·temp·platt) 각각** 마진 적합(val micro) → test 1회 적용

In [7]:
def eval_decode(Lval, Ltest, prob_fn, base_test):
    grid = np.round(np.linspace(0.0, 0.05, 26), 4)
    bm, bf = max(((float(x), micro(Yval, decode(Lval, x, prob_fn)[0])) for x in grid), key=lambda t: t[1])
    pred, kc = decode(Ltest, bm, prob_fn)
    mt = micro(Ytest, pred)
    return {"margin": bm, "val_micro": round(bf, 4), "test_micro": round(mt, 4),
            "delta_pt": round(100 * (mt - base_test), 3),
            "slice": slice_report(Ytest, pred, ktest),
            "k_dist": np.bincount(kc)[:11].tolist()}

def run_model(m):
    tag, kind = m["tag"], m["kind"]
    Lval, Ltest = load_logits(tag, "val", len(Yval)), load_logits(tag, "test", len(Ytest))
    CAL, val_bce, params = fit_calibration(Lval, Yval)

    diag = {"val": sump_diag(Lval, kval, CAL), "test": sump_diag(Ltest, ktest, CAL)}
    base_test = micro(Ytest, Ltest >= 0)

    variants = {cn: eval_decode(Lval, Ltest, CAL[cn], base_test) for cn in CAL_NAMES}   # raw·temp·platt
    return {
        "tag": tag, "kind": kind,
        "calibration": {"val_bce": val_bce, "best_bce": min(val_bce, key=val_bce.get), **params},
        "diagnostic": diag,
        "baseline": {"val_micro": round(micro(Yval, Lval >= 0), 4), "test_micro": round(base_test, 4),
                     "slice": slice_report(Ytest, Ltest >= 0, ktest)},
        "oracle_k_ceiling": round(oracle_k(Ltest, Ytest), 4),
        "variants": variants,
    }

RESULTS = {m["kind"]: run_model(m) for m in MODELS}
print("완료:", list(RESULTS))

완료: ['focal', 'bce']


In [8]:
for kind, r in RESULTS.items():
    c, dt = r["calibration"], r["diagnostic"]["test"]["k>=2"]
    base, ceil = r["baseline"], r["oracle_k_ceiling"]
    print(f"\n########## {kind.upper()}  ({r['tag']}) ##########")
    print(f"T={c['temp_T']}  platt(a={c['platt_a']}, b={c['platt_b']})  best BCE={c['best_bce']}")
    print(f"k>=2 Σp  raw {dt['raw']:.3f} | temp {dt['temp']:.3f} | platt {dt['platt']:.3f}   (gold 2.355)")
    bt = base["test_micro"]
    print(f"baseline(τ=0.5) test micro {bt:.4f}  ·  오라클-k 상한 {ceil:.4f} (+{100*(ceil-bt):.2f}pt)")
    print(f"  {'확률원':<7}{'margin':>8}{'test micro':>12}{'Δpt':>8}{'k=1 micro':>11}{'k>=2 micro':>12}{'k>=2 mean_pred':>16}")
    print(f"  {'base':<7}{'-':>8}{bt:>12.4f}{'-':>8}{base['slice']['k=1']['micro']:>11.4f}"
          f"{base['slice']['k>=2']['micro']:>12.4f}{base['slice']['k>=2']['mean_pred']:>16.3f}")
    for cn in CAL_NAMES:
        var = r["variants"][cn]; s = var["slice"]
        print(f"  {cn:<7}{var['margin']:>8}{var['test_micro']:>12.4f}{var['delta_pt']:>+8.2f}"
              f"{s['k=1']['micro']:>11.4f}{s['k>=2']['micro']:>12.4f}{s['k>=2']['mean_pred']:>16.3f}")


########## FOCAL  (modernbert-patent-len512) ##########
T=0.9257  platt(a=0.972, b=-0.8307)  best BCE=platt
k>=2 Σp  raw 2.248 | temp 2.133 | platt 1.826   (gold 2.355)
baseline(τ=0.5) test micro 0.8601  ·  오라클-k 상한 0.8760 (+1.59pt)
  확률원      margin  test micro     Δpt  k=1 micro  k>=2 micro  k>=2 mean_pred
  base          -      0.8601       -     0.8822      0.7994           1.930
  raw        0.05      0.8579   -0.22     0.8760      0.8084           2.038
  temp      0.048      0.8585   -0.16     0.8772      0.8072           2.020
  platt     0.006      0.8577   -0.24     0.8829      0.7871           1.862

########## BCE  (modernbert-patent-len512-bce) ##########
T=1.4954  platt(a=0.6175, b=-0.6707)  best BCE=platt
k>=2 Σp  raw 1.903 | temp 2.045 | platt 1.782   (gold 2.355)
baseline(τ=0.5) test micro 0.8538  ·  오라클-k 상한 0.8693 (+1.55pt)
  확률원      margin  test micro     Δpt  k=1 micro  k>=2 micro  k>=2 mean_pred
  base          -      0.8538       -     0.8782      0.7864       

## 4. focal — 세 확률원 대조 (부호 뒤집힘 확인)

baseline vs raw·temp·platt를 대조. raw·temp는 k≥2를 회수(신호가 미보정 확률에 존재)하나 k=1을 과대예측하고, platt은 전역 BCE를 맞추느라 양쪽을 눌러 k≥2까지 죽인다.

In [9]:
r = RESULTS["focal"]
cols = ["baseline"] + CAL_NAMES
def val_of(col, path):
    s = r["baseline"]["slice"] if col == "baseline" else r["variants"][col]["slice"]
    lab, key = path
    return s[lab][key]

print("FOCAL — baseline vs raw · temp · platt\n")
print(f"{'':<18}" + "".join(f"{c:>11}" for c in cols))
print("-" * (18 + 11 * len(cols)))
gm = [r["baseline"]["test_micro"]] + [r["variants"][c]["test_micro"] for c in CAL_NAMES]
print(f"{'global micro':<18}" + "".join(f"{v:>11.4f}" for v in gm))
for lab in ("k=1", "k>=2"):
    for key, fmt in [("micro", "{:>11.4f}"), ("fp", "{:>11,}"), ("mean_pred", "{:>11.3f}")]:
        print(f"{lab+' '+key:<18}" + "".join(fmt.format(val_of(c, (lab, key))) for c in cols))

k2b = r["baseline"]["slice"]["k>=2"]["micro"]
print("\n확률원별 k>=2 micro:  " + " · ".join(f"{c} {r['variants'][c]['slice']['k>=2']['micro']:.4f}"
                                            f"({100*(r['variants'][c]['slice']['k>=2']['micro']-k2b):+.2f})" for c in CAL_NAMES))
print("→ raw·temp는 k≥2 회수(+), platt은 신호 파괴(−). 전역 micro는 셋 다 음성 = 부호 뒤집힘")

FOCAL — baseline vs raw · temp · platt

                     baseline        raw       temp      platt
--------------------------------------------------------------
global micro           0.8601     0.8579     0.8585     0.8577
k=1 micro              0.8822     0.8760     0.8772     0.8829
k=1 fp                  1,549      1,855      1,817      1,618
k=1 mean_pred           1.079      1.124      1.119      1.093
k>=2 micro             0.7994     0.8084     0.8072     0.7871
k>=2 fp                   367        444        430        342
k>=2 mean_pred          1.930      2.038      2.020      1.862

확률원별 k>=2 micro:  raw 0.8084(+0.90) · temp 0.8072(+0.78) · platt 0.7871(-1.23)
→ raw·temp는 k≥2 회수(+), platt은 신호 파괴(−). 전역 micro는 셋 다 음성 = 부호 뒤집힘


## 저장

결과를 `output/cardinality_decoding_test.json`에 SSOT로 저장(두 모델 × 세 확률원 병기).

In [10]:
result = {
    "num_labels": NUM, "kmax": config["kmax"], "split_fit": "val", "split_apply": "test",
    "cal_names": CAL_NAMES, "models": RESULTS,
    "verdict": ("negative(전역 micro) — 2번째 양성 신호는 미보정 확률에 존재하고(focal k>=2 Σp_raw 2.25≈2.355) "
                "raw·temp 디코딩이 k>=2 slice를 회수(focal +0.90pt/+0.75pt, mean_pred→2.0)하나, 같은 확장이 "
                "k=1(85%)을 과대예측해 전역 micro는 음성(focal raw -0.22 / temp -0.16 / platt -0.24). "
                "platt은 전역 BCE를 맞추느라 신호를 눌러 k>=2까지 죽인다. proper BCE도 동형. "
                "= ADR-0009 FP:FN 부호 뒤집힘의 디코딩 수준 재확인 — 문서별 k(오라클-k, 닫힘)만이 분리 가능."),
}
fp = OUT / "cardinality_decoding_test.json"
fp.write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding="utf-8")
print("saved:", fp)

saved: C:\workspace\patent_disc\output\cardinality_decoding_test.json


## 결론

전역 micro 기준 음성

- **raw**: focal k≥2 `Σp_raw` 2.25 ≈ 정답 2.355. raw 디코딩은 **k≥2 slice를 +0.90pt 회수**(mean_pred 1.93→2.04, 눌린 2번째 양성 107건 복원). 그러나 k=1도 부풀려(Σp 1.25>1.0) 과대예측(FP +306) → 전역 −0.22pt.
- **temperature**(T=0.926, sharpen): raw와 platt 사이. focal에서 셋 중 **최선(−0.16pt)**이나 여전히 음성 — 스케일만으로는 k=1/k≥2를 분리 못 한다. BCE는 T>1(평탄화)이라 음성 질량을 부풀려 오히려 더 나쁘다(−0.27pt).
- **platt**(스케일+이동): 전역 BCE(문서당 음성 ≈187개 지배)를 맞추는 하향 이동이 k≥2 양성 질량을 1.83으로 눌러 **신호를 지운다** → k≥2가 오히려 하락(−0.24pt).

즉 2번째 양성 신호는 존재하고 미보정 확률에서 회수 가능하나, 전역(문서-Σp) 규칙으로는 k=1 과대예측과 분리하지 못했다.